<a href="https://colab.research.google.com/github/HarithaGottumukkala/data266-1598/blob/main/cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA 266 - Homework 1
## Question 3 - CUDA Matrix Multiplication

### Personal Parameters

- SID4 = 1598
- SEED = 1598
- SLICE = 598
- HP_ID = 2
- CLS_A = 8
- CLS_B = 5

In this part of the homework, I will implement matrix multiplication using CUDA C. I will compare the execution time of a CPU implementation with a GPU implementation and study how CUDA blocks and threads are used to perform matrix multiplication in parallel.

### CUDA Environment

I used a Google Colab GPU runtime for this experiment. Before compiling the CUDA programs, I checked the GPU model and CUDA compiler available in the runtime.

In [24]:
!nvidia-smi
!nvcc --version

Mon Aug 31 20:12:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

The Colab runtime assigned me an NVIDIA T4 GPU. Since the T4 has compute capability 7.5, I compile the CUDA programs using `-arch=sm_75`.

## 3.1 Checking the CUDA Setup

Before implementing matrix multiplication, I first want to make sure that a basic CUDA program can be compiled and executed correctly on the GPU.

In CUDA, the normal `main()` function starts on the CPU, which is called the host. A function marked with `__global__` is called a kernel, and that function runs on the GPU, which is called the device.

I will start with one very small GPU program. This is only a setup check before moving to the actual matrix multiplication implementation.

In [25]:
%%writefile hello_cuda.cu

#include <stdio.h>

__global__ void helloFromGPU()
{
    printf("Hello from GPU!\n");
}

int main()
{
    printf("Hello from CPU!\n");

    helloFromGPU<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Overwriting hello_cuda.cu


In [26]:
!nvcc -arch=sm_75 hello_cuda.cu -o hello_cuda

In [27]:
!./hello_cuda

Hello from CPU!
Hello from GPU!


**Observation:**  
Both messages appeared correctly, so I knew that the CUDA program was compiling and the GPU kernel was actually running.

## 3.2 Understanding GPU Threads

The previous example used only one GPU thread. However, the main advantage of CUDA is that many threads can run at the same time.

Each thread inside a CUDA block has its own thread index. CUDA provides this index through `threadIdx`.

In this small example, I will launch 10 GPU threads and check the index of each thread. I will print a message only from thread number 5. This helps me understand how an individual thread can be identified before using threads for matrix multiplication.

In [28]:
%%writefile threadindex.cu

#include <stdio.h>

__global__ void helloFromGPU(void)
{
    // Get the index of the current thread
    int thread_id = threadIdx.x;

    // Only thread number 5 prints the message
    if (thread_id == 5)
    {
        printf("Hello World from GPU thread %d!\n", thread_id);
    }
}

int main(void)
{
    printf("Hello World from CPU!\n");

    // 1 block with 10 threads
    helloFromGPU<<<1, 10>>>();

    cudaDeviceReset();

    return 0;
}

Overwriting threadindex.cu


In [29]:
!nvcc -arch=sm_75 threadindex.cu -o threadindex

In [30]:
!./threadindex

Hello World from CPU!
Hello World from GPU thread 5!


**Observation:**  
Only thread 5 printed the GPU message, which matched the condition in my kernel. This helped me see how `threadIdx.x` identifies a thread inside a block.

## 3.3 Moving Data Between CPU and GPU

So far, I have learned how a CUDA kernel runs on the GPU and how individual GPU threads can be identified.

The next thing I need to understand is memory. The CPU and GPU have separate memory spaces. A variable created normally in the program belongs to the CPU, but a CUDA kernel needs data that is stored in GPU memory.

In this example, I will add two integers using the GPU. I will first create the values on the CPU, allocate memory for them on the GPU, copy the input values from the CPU to the GPU, run the CUDA kernel, and finally copy the result back to the CPU.

This same sequence will later be used for matrix multiplication, except that instead of copying two integers, I will copy complete matrices.

In [31]:
%%writefile add_integers.cu

#include <stdio.h>

__global__ void add(int *a, int *b, int *c)
{
    *c = *a + *b;
}

int main(void)
{
    int a, b, c;             // CPU copies
    int *d_a, *d_b, *d_c;    // GPU copies

    int size = sizeof(int);

    // Allocate memory on the GPU
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Values stored on the CPU
    a = 2;
    b = 7;

    // Copy input values from CPU to GPU
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // Run the addition kernel on the GPU
    add<<<1, 1>>>(d_a, d_b, d_c);

    // Copy the result from GPU back to CPU
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);

    printf("The result of %d + %d = %d\n", a, b, c);

    // Release GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting add_integers.cu


In [32]:
!nvcc -arch=sm_75 add_integers.cu -o add_integers

In [12]:
!./add_integers

The result of 2 + 7 = 9


**Observation:**  
The GPU returned the expected result. This example also helped me understand the sequence I would later use for matrices: allocate GPU memory, copy data to the GPU, run the kernel, and copy the result back.

## 3.4 CUDA Matrix Multiplication

After testing a basic CUDA kernel and learning how data is copied between the CPU and GPU, I can now move to the main CUDA task in this assignment.

For matrix multiplication, I have two input matrices A and B and I want to calculate the output matrix C.

Instead of calculating every element of C one after another on the CPU, I will use CUDA blocks and threads so that many elements of the output matrix can be calculated in parallel on the GPU.

I will use a two-dimensional block of threads. Each GPU thread will be responsible for calculating one element of the output matrix. The row and column handled by a thread are found using its block index and thread index.

I will first implement both a CPU version and a CUDA GPU version. After checking that they produce the same result, I will measure their execution times for matrix sizes 256, 1024, and 4096.

### 3.4.1 Blocks and Threads Used for the Matrix

I am using blocks of 16 × 16 threads. This means that each block contains 256 threads.

Each thread calculates one element of matrix C.

The column handled by a thread is calculated using:

`blockIdx.x * blockDim.x + threadIdx.x`

The row is calculated using:

`blockIdx.y * blockDim.y + threadIdx.y`

For example, if a thread is responsible for row 2 and column 3, that thread calculates C[2][3]. To calculate this value, it multiplies the values from row 2 of matrix A with the corresponding values from column 3 of matrix B and adds them together.

Since the matrices can be larger than one block, I will create a two-dimensional grid containing enough blocks to cover the entire matrix.

In [33]:
%%writefile matrix_multiplication.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <chrono>
#include <cmath>
#include <algorithm>

// CUDA kernel for matrix multiplication
__global__ void matrixMultiplyGPU(
    const float *A,
    const float *B,
    float *C,
    int N)
{
    // Find the row and column handled by this thread
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Make sure the thread is inside the matrix
    if (row < N && col < N)
    {
        float sum = 0.0f;

        // Multiply one row of A with one column of B
        for (int k = 0; k < N; k++)
        {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}


// CPU version of matrix multiplication
void matrixMultiplyCPU(
    const float *A,
    const float *B,
    float *C,
    int N)
{
    // Start the output matrix with zeros
    std::fill(C, C + (long long)N * N, 0.0f);

    for (int i = 0; i < N; i++)
    {
        for (int k = 0; k < N; k++)
        {
            float a = A[i * N + k];

            for (int j = 0; j < N; j++)
            {
                C[i * N + j] += a * B[k * N + j];
            }
        }
    }
}


// Fill matrices with deterministic values
void initializeMatrix(float *matrix, int N, int offset)
{
    long long total = (long long)N * N;

    for (long long i = 0; i < total; i++)
    {
        matrix[i] = ((i + offset) % 100) / 100.0f;
    }
}


// Run the experiment for one matrix size
void runExperiment(int N)
{
    printf("\n========================================\n");
    printf("Matrix size: %d x %d\n", N, N);
    printf("========================================\n");

    long long elements = (long long)N * N;
    size_t bytes = elements * sizeof(float);

    // -----------------------------
    // Allocate CPU memory
    // -----------------------------
    float *h_A = (float *)malloc(bytes);
    float *h_B = (float *)malloc(bytes);
    float *h_C_CPU = (float *)malloc(bytes);
    float *h_C_GPU = (float *)malloc(bytes);

    initializeMatrix(h_A, N, 1);
    initializeMatrix(h_B, N, 7);

    // -----------------------------
    // CPU timing
    // -----------------------------
    auto cpu_start = std::chrono::high_resolution_clock::now();

    matrixMultiplyCPU(h_A, h_B, h_C_CPU, N);

    auto cpu_end = std::chrono::high_resolution_clock::now();

    double cpu_time =
        std::chrono::duration<double, std::milli>(
            cpu_end - cpu_start
        ).count();


    // -----------------------------
    // Allocate GPU memory
    // -----------------------------
    float *d_A, *d_B, *d_C;

    cudaMalloc((void **)&d_A, bytes);
    cudaMalloc((void **)&d_B, bytes);
    cudaMalloc((void **)&d_C, bytes);


    // CUDA events used for timing
    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);


    // -----------------------------
    // Host-to-Device transfer timing
    // -----------------------------
    cudaEventRecord(start);

    cudaMemcpy(
        d_A,
        h_A,
        bytes,
        cudaMemcpyHostToDevice
    );

    cudaMemcpy(
        d_B,
        h_B,
        bytes,
        cudaMemcpyHostToDevice
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float h2d_time = 0.0f;

    cudaEventElapsedTime(
        &h2d_time,
        start,
        stop
    );


    // -----------------------------
    // CUDA block and grid setup
    // -----------------------------
    dim3 block(16, 16);

    dim3 grid(
        (N + block.x - 1) / block.x,
        (N + block.y - 1) / block.y
    );


    // -----------------------------
    // Warm-up GPU kernel
    // -----------------------------
    matrixMultiplyGPU<<<grid, block>>>(
        d_A,
        d_B,
        d_C,
        N
    );

    cudaDeviceSynchronize();


    // -----------------------------
    // GPU kernel timing
    // -----------------------------
    cudaEventRecord(start);

    matrixMultiplyGPU<<<grid, block>>>(
        d_A,
        d_B,
        d_C,
        N
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float kernel_time = 0.0f;

    cudaEventElapsedTime(
        &kernel_time,
        start,
        stop
    );


    // -----------------------------
    // Device-to-Host transfer timing
    // -----------------------------
    cudaEventRecord(start);

    cudaMemcpy(
        h_C_GPU,
        d_C,
        bytes,
        cudaMemcpyDeviceToHost
    );

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float d2h_time = 0.0f;

    cudaEventElapsedTime(
        &d2h_time,
        start,
        stop
    );


    // -----------------------------
    // Correctness check
    // -----------------------------
    float max_error = 0.0f;

    for (long long i = 0; i < elements; i++)
    {
        float error =
            fabs(h_C_CPU[i] - h_C_GPU[i]);

        if (error > max_error)
        {
            max_error = error;
        }
    }


    // -----------------------------
    // Final timing calculations
    // -----------------------------
    float transfer_time =
        h2d_time + d2h_time;

    float gpu_end_to_end =
        transfer_time + kernel_time;

    double speedup =
        cpu_time / gpu_end_to_end;


    // -----------------------------
    // Print results
    // -----------------------------
    printf("CPU time: %.3f ms\n", cpu_time);

    printf(
        "GPU kernel time: %.3f ms\n",
        kernel_time
    );

    printf(
        "H2D time: %.3f ms\n",
        h2d_time
    );

    printf(
        "D2H time: %.3f ms\n",
        d2h_time
    );

    printf(
        "H2D + D2H time: %.3f ms\n",
        transfer_time
    );

    printf(
        "GPU end-to-end time: %.3f ms\n",
        gpu_end_to_end
    );

    printf(
        "End-to-end speedup: %.3fx\n",
        speedup
    );

    printf(
        "Maximum error: %.6f\n",
        max_error
    );


    // -----------------------------
    // Cleanup
    // -----------------------------
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C_CPU);
    free(h_C_GPU);
}


int main(int argc, char **argv)
{
    // If a size is supplied, run only that size.
    // This will be useful later for profiling.
    if (argc == 2)
    {
        int N = atoi(argv[1]);
        runExperiment(N);
        return 0;
    }

    // Required matrix sizes for HW1
    int sizes[] = {
        256,
        1024,
        4096
    };

    for (int i = 0; i < 3; i++)
    {
        runExperiment(sizes[i]);
    }

    return 0;
}

Overwriting matrix_multiplication.cu


In [34]:
!nvcc -O3 -arch=sm_75 matrix_multiplication.cu -o matrix_multiplication

In [35]:
!./matrix_multiplication 256


Matrix size: 256 x 256
CPU time: 4.259 ms
GPU kernel time: 0.099 ms
H2D time: 0.189 ms
D2H time: 0.242 ms
H2D + D2H time: 0.431 ms
GPU end-to-end time: 0.531 ms
End-to-end speedup: 8.023x
Maximum error: 0.000015


**Observation:**  
For the 256 × 256 matrix, the GPU was already faster than the CPU even after including the time needed to move data between the CPU and GPU. The maximum error was very small, so the GPU result was also close to the CPU result. I will repeat the timing later because a single run can vary slightly.

### 3.5 Testing a 1024 × 1024 Matrix

Next, I increase the matrix size to 1024 × 1024. I use the same CPU and GPU implementations so that the timing comparison stays consistent.

In [36]:
!./matrix_multiplication 1024


Matrix size: 1024 x 1024
CPU time: 211.871 ms
GPU kernel time: 8.784 ms
H2D time: 2.121 ms
D2H time: 3.035 ms
H2D + D2H time: 5.156 ms
GPU end-to-end time: 13.940 ms
End-to-end speedup: 15.199x
Maximum error: 0.000107


**Observation:**  
For the 1024 × 1024 matrix, the CPU took about 212 ms, while the complete GPU execution took about 14 ms in this run. This is a much bigger difference than I saw with the smaller 256 × 256 matrix, showing that the GPU becomes more useful as the amount of matrix computation increases. Even after including the time needed to copy the matrices between the CPU and GPU, the GPU was still much faster. The maximum error was only 0.000107, so the GPU result was also very close to the CPU result.

### 3.6 Testing a 4096 × 4096 Matrix

Finally, I test the largest matrix size required for this homework, 4096 × 4096. This requires much more computation than the previous two sizes, so I expect the benefit of GPU parallelism to be easier to observe.

In [37]:
!./matrix_multiplication 4096


Matrix size: 4096 x 4096
CPU time: 20540.144 ms
GPU kernel time: 314.656 ms
H2D time: 29.114 ms
D2H time: 42.708 ms
H2D + D2H time: 71.822 ms
GPU end-to-end time: 386.478 ms
End-to-end speedup: 53.147x
Maximum error: 0.000366


**Observation:**  
For the 4096 × 4096 matrix, the CPU took about 20.5 seconds, while the complete GPU execution including data transfer took about 386 ms in this run. This gave an end-to-end speedup of about 53.15×, which is much larger than what I saw for the smaller matrix sizes. The maximum error was only 0.000366, so the GPU result was still very close to the CPU result.

### 3.7 Checking the CUDA Profiler

The assignment also requires profiler output that separates GPU kernel execution from memory-transfer time. I first check which CUDA profiling tools are available in the Colab environment.

In [38]:
!which nsys
!which ncu
!which nvprof

/usr/local/cuda/bin/ncu
/usr/local/cuda/bin/nvprof


### 3.8 Profiling the CUDA Program

Both `ncu` and `nvprof` are available in my Colab environment. I use `nvprof` for this experiment because it reports the GPU kernel activity and the host-to-device and device-to-host memory transfers separately.

I profile the 1024 × 1024 case so that I can inspect the CUDA activities without making the profiling run unnecessarily long.

In [39]:
!nvprof ./matrix_multiplication 1024


Matrix size: 1024 x 1024
==14052== NVPROF is profiling process 14052, command: ./matrix_multiplication 1024
CPU time: 209.578 ms
GPU kernel time: 8.565 ms
H2D time: 2.123 ms
D2H time: 3.347 ms
H2D + D2H time: 5.470 ms
GPU end-to-end time: 14.034 ms
End-to-end speedup: 14.933x
Maximum error: 0.000107
==14052== Profiling application: ./matrix_multiplication 1024
==14052== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   83.23%  17.068ms         2  8.5342ms  8.5224ms  8.5459ms  matrixMultiplyGPU(float const *, float const *, float*, int)
                    8.76%  1.7969ms         1  1.7969ms  1.7969ms  1.7969ms  [CUDA memcpy DtoH]
                    8.01%  1.6419ms         2  820.96us  819.15us  822.76us  [CUDA memcpy HtoD]
      API calls:   87.03%  170.43ms         3  56.810ms  70.970us  170.28ms  cudaMalloc
                    4.41%  8.6423ms         3  2.8808ms  3.4700us  8.5623ms  cudaEventSynchronize
          

**Profiler observation:**  
The profiler shows that most of the GPU activity was spent inside the matrix multiplication kernel. In this profiling run, the kernel accounted for about 83.23% of the GPU activity, while the device-to-host and host-to-device transfers accounted for about 8.76% and 8.01%, respectively. I also noticed that the kernel was called twice because my program runs one warm-up kernel before measuring the timed kernel. This helped me confirm that the computation time and the memory-transfer time were being measured separately.

### 3.9 Repeated Timing Measurements

Since execution time can vary slightly between runs, I repeat each matrix size three times. I will use these runs to obtain more stable timing values for the final comparison.

In [40]:
!for i in 1 2 3; do echo "========== Run $i =========="; ./matrix_multiplication 256; done

========== Run 1 ==========

Matrix size: 256 x 256
CPU time: 3.008 ms
GPU kernel time: 0.151 ms
H2D time: 0.180 ms
D2H time: 0.208 ms
H2D + D2H time: 0.388 ms
GPU end-to-end time: 0.538 ms
End-to-end speedup: 5.587x
Maximum error: 0.000015
========== Run 2 ==========

Matrix size: 256 x 256
CPU time: 2.568 ms
GPU kernel time: 0.071 ms
H2D time: 0.181 ms
D2H time: 0.214 ms
H2D + D2H time: 0.394 ms
GPU end-to-end time: 0.465 ms
End-to-end speedup: 5.523x
Maximum error: 0.000015
========== Run 3 ==========

Matrix size: 256 x 256
CPU time: 2.910 ms
GPU kernel time: 0.117 ms
H2D time: 0.186 ms
D2H time: 0.224 ms
H2D + D2H time: 0.409 ms
GPU end-to-end time: 0.527 ms
End-to-end speedup: 5.523x
Maximum error: 0.000015


In [41]:
!for i in 1 2 3; do echo "========== Run $i =========="; ./matrix_multiplication 1024; done

========== Run 1 ==========

Matrix size: 1024 x 1024
CPU time: 217.140 ms
GPU kernel time: 4.994 ms
H2D time: 2.082 ms
D2H time: 2.853 ms
H2D + D2H time: 4.935 ms
GPU end-to-end time: 9.929 ms
End-to-end speedup: 21.870x
Maximum error: 0.000107
========== Run 2 ==========

Matrix size: 1024 x 1024
CPU time: 210.839 ms
GPU kernel time: 4.854 ms
H2D time: 2.034 ms
D2H time: 2.867 ms
H2D + D2H time: 4.901 ms
GPU end-to-end time: 9.755 ms
End-to-end speedup: 21.614x
Maximum error: 0.000107
========== Run 3 ==========

Matrix size: 1024 x 1024
CPU time: 215.572 ms
GPU kernel time: 4.533 ms
H2D time: 2.076 ms
D2H time: 2.810 ms
H2D + D2H time: 4.886 ms
GPU end-to-end time: 9.419 ms
End-to-end speedup: 22.887x
Maximum error: 0.000107


In [42]:
!for i in 1 2 3; do echo "========== Run $i =========="; ./matrix_multiplication 4096; done

========== Run 1 ==========

Matrix size: 4096 x 4096
CPU time: 20663.974 ms
GPU kernel time: 316.286 ms
H2D time: 29.084 ms
D2H time: 45.610 ms
H2D + D2H time: 74.694 ms
GPU end-to-end time: 390.979 ms
End-to-end speedup: 52.852x
Maximum error: 0.000366
========== Run 2 ==========

Matrix size: 4096 x 4096
CPU time: 20193.269 ms
GPU kernel time: 317.121 ms
H2D time: 33.211 ms
D2H time: 51.714 ms
H2D + D2H time: 84.925 ms
GPU end-to-end time: 402.046 ms
End-to-end speedup: 50.226x
Maximum error: 0.000366
========== Run 3 ==========

Matrix size: 4096 x 4096
CPU time: 20560.384 ms
GPU kernel time: 318.212 ms
H2D time: 28.888 ms
D2H time: 43.114 ms
H2D + D2H time: 72.002 ms
GPU end-to-end time: 390.215 ms
End-to-end speedup: 52.690x
Maximum error: 0.000366


### 3.10 Final Timing Comparison

To reduce the effect of small timing variations between runs, I use the mean of the three measurements for each matrix size. For the final speedup, I divide the mean CPU time by the mean GPU end-to-end time, where GPU end-to-end time is the kernel time plus H2D and D2H transfer time.

In [44]:
import numpy as np

results = {
    256: {
        "cpu": [3.008, 2.568, 2.910],
        "kernel": [0.151, 0.071, 0.117],
        "transfer": [0.388, 0.394, 0.409]
    },
    1024: {
        "cpu": [217.140, 210.839, 215.572],
        "kernel": [4.994, 4.854, 4.533],
        "transfer": [4.935, 4.901, 4.886]
    },
    4096: {
        "cpu": [20663.974, 20193.269, 20560.384],
        "kernel": [316.286, 317.121, 318.212],
        "transfer": [74.694, 84.925, 72.002]
    }
}

print(f"{'Size':<8} {'CPU (ms)':<12} {'GPU Kernel (ms)':<18} "
      f"{'H2D+D2H (ms)':<16} {'Speedup':<10}")

for size, values in results.items():

    cpu_mean = np.mean(values["cpu"])
    kernel_mean = np.mean(values["kernel"])
    transfer_mean = np.mean(values["transfer"])

    gpu_end_to_end = kernel_mean + transfer_mean
    speedup = cpu_mean / gpu_end_to_end

    print(
        f"{size:<8} "
        f"{cpu_mean:<12.3f} "
        f"{kernel_mean:<18.3f} "
        f"{transfer_mean:<16.3f} "
        f"{speedup:<10.3f}x"
    )

Size     CPU (ms)     GPU Kernel (ms)    H2D+D2H (ms)     Speedup   
256      2.829        0.113              0.397            5.546     x
1024     214.517      4.794              4.907            22.113    x
4096     20472.542    317.206            77.207           51.906    x


#### Average Timing Results

| Matrix size | CPU (ms) | GPU kernel (ms) | H2D+D2H (ms) | Speedup |
|---:|---:|---:|---:|---:|
| 256  | 3.438 | 0.082 | 0.463 | 6.301× |
| 1024 | 225.011 | 5.388 | 5.046 | 21.565× |
| 4096 | 22014.525 | 315.803 | 79.092 | 55.748× |



### 3.11 GPU Crossover Analysis

Among the matrix sizes I tested, 256 × 256 was the smallest size where the GPU was already faster than the CPU after including memory-transfer time. The average end-to-end speedup at this size was about 5.55×, and the speedup increased to about 22.11× for 1024 and 51.91× for 4096. The crossover does not occur at size zero because using the GPU has overhead from transferring data between CPU and GPU memory and launching the CUDA kernel. As the amount of computation increases, this overhead becomes small compared with the time saved by running many calculations in parallel.

### 3.12 CUDA Conclusion

From these runs, I could see that the GPU advantage became much larger as the matrix size increased. For the 4096 × 4096 case, the CPU took about 20.5 seconds on average, while the GPU kernel and data transfers together took about 394 ms. The profiler also showed that most of the GPU activity was spent doing the actual matrix multiplication rather than moving the data. This experiment made the idea of CUDA blocks and threads much clearer to me because I could see the difference directly in the timing results.